In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

import warnings
warnings.filterwarnings("ignore")

In [2]:
matches = pd.read_csv("../data/matches.csv")

matches.head()

,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


In [3]:
matches.info()


<class 'pandas.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               756 non-null    int64
 1   season           756 non-null    int64
 2   city             749 non-null    str  
 3   date             756 non-null    str  
 4   team1            756 non-null    str  
 5   team2            756 non-null    str  
 6   toss_winner      756 non-null    str  
 7   toss_decision    756 non-null    str  
 8   result           756 non-null    str  
 9   dl_applied       756 non-null    int64
 10  winner           752 non-null    str  
 11  win_by_runs      756 non-null    int64
 12  win_by_wickets   756 non-null    int64
 13  player_of_match  752 non-null    str  
 14  venue            756 non-null    str  
 15  umpire1          754 non-null    str  
 16  umpire2          754 non-null    str  
 17  umpire3          119 non-null    str  
dtypes: int64(5), str(13)


In [4]:
matches["winner"].value_counts()

winner
Mumbai Indians                 109
Chennai Super Kings            100
Kolkata Knight Riders           92
Royal Challengers Bangalore     84
Kings XI Punjab                 82
Rajasthan Royals                75
Delhi Daredevils                67
Sunrisers Hyderabad             58
Deccan Chargers                 29
Gujarat Lions                   13
Pune Warriors                   12
Rising Pune Supergiant          10
Delhi Capitals                  10
Kochi Tuskers Kerala             6
Rising Pune Supergiants          5
Name: count, dtype: int64

In [5]:
matches["team1"].unique()

<ArrowStringArray>
[        'Sunrisers Hyderabad',              'Mumbai Indians',
               'Gujarat Lions',      'Rising Pune Supergiant',
 'Royal Challengers Bangalore',       'Kolkata Knight Riders',
            'Delhi Daredevils',             'Kings XI Punjab',
         'Chennai Super Kings',            'Rajasthan Royals',
             'Deccan Chargers',        'Kochi Tuskers Kerala',
               'Pune Warriors',     'Rising Pune Supergiants',
              'Delhi Capitals']
Length: 15, dtype: str

In [6]:
matches["team2"].unique()

<ArrowStringArray>
['Royal Challengers Bangalore',      'Rising Pune Supergiant',
       'Kolkata Knight Riders',             'Kings XI Punjab',
            'Delhi Daredevils',         'Sunrisers Hyderabad',
              'Mumbai Indians',               'Gujarat Lions',
            'Rajasthan Royals',         'Chennai Super Kings',
             'Deccan Chargers',               'Pune Warriors',
        'Kochi Tuskers Kerala',     'Rising Pune Supergiants',
              'Delhi Capitals']
Length: 15, dtype: str

In [7]:
team_replace = {

    "Delhi Daredevils": "Delhi Capitals",

    "Kings XI Punjab": "Punjab Kings",

    "Rising Pune Supergiants": "Rising Pune Supergiants",

    "Rising Pune Supergiant": "Rising Pune Supergiants"
}


matches["team1"] = matches["team1"].replace(team_replace)

matches["team2"] = matches["team2"].replace(team_replace)

matches["winner"] = matches["winner"].replace(team_replace)

matches["toss_winner"] = matches["toss_winner"].replace(team_replace)

In [8]:
matches = matches.dropna(
    subset=["winner"]
)

matches.shape

(752, 18)

In [9]:
data = matches[
    [
        "team1",
        "team2",
        "venue",
        "toss_winner",
        "toss_decision",
        "winner"
    ]
]


data.head()

,team1,team2,venue,toss_winner,toss_decision,winner
0,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,Sunrisers Hyderabad
1,Mumbai Indians,Rising Pune Supergiants,Maharashtra Cricket Association Stadium,Rising Pune Supergiants,field,Rising Pune Supergiants
2,Gujarat Lions,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,field,Kolkata Knight Riders
3,Rising Pune Supergiants,Punjab Kings,Holkar Cricket Stadium,Punjab Kings,field,Punjab Kings
4,Royal Challengers Bangalore,Delhi Capitals,M Chinnaswamy Stadium,Royal Challengers Bangalore,bat,Royal Challengers Bangalore


In [10]:
X = data.drop("winner", axis=1)

y = data["winner"]


X.head()

,team1,team2,venue,toss_winner,toss_decision
0,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field
1,Mumbai Indians,Rising Pune Supergiants,Maharashtra Cricket Association Stadium,Rising Pune Supergiants,field
2,Gujarat Lions,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,field
3,Rising Pune Supergiants,Punjab Kings,Holkar Cricket Stadium,Punjab Kings,field
4,Royal Challengers Bangalore,Delhi Capitals,M Chinnaswamy Stadium,Royal Challengers Bangalore,bat


In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


categorical_features = [
    "team1",
    "team2",
    "venue",
    "toss_winner",
    "toss_decision"
]


encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)


X_encoded = encoder.fit_transform(X)


X_encoded.shape

(752, 82)

In [12]:
from sklearn.preprocessing import LabelEncoder


winner_encoder = LabelEncoder()


y_encoded = winner_encoder.fit_transform(y)


winner_encoder.classes_

array(['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals',
       'Gujarat Lions', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders',
       'Mumbai Indians', 'Pune Warriors', 'Punjab Kings',
       'Rajasthan Royals', 'Rising Pune Supergiants',
       'Royal Challengers Bangalore', 'Sunrisers Hyderabad'], dtype=object)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [14]:
lr_model = LogisticRegression(
    max_iter=1000
)

lr_model.fit(
    X_train,
    y_train
)


lr_pred = lr_model.predict(
    X_test
)


lr_accuracy = accuracy_score(
    y_test,
    lr_pred
)


print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.5894039735099338


In [15]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


rf_model.fit(
    X_train,
    y_train
)


rf_pred = rf_model.predict(
    X_test
)


rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)


print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.5695364238410596


In [16]:
results = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "Random Forest"
        ],
        "Accuracy": [
            lr_accuracy,
            rf_accuracy
        ]
    }
)


results

,Model,Accuracy
0,Logistic Regression,0.589404
1,Random Forest,0.569536


In [17]:
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=winner_encoder.classes_
    )
)

                             precision    recall  f1-score   support

        Chennai Super Kings       0.59      0.65      0.62        20
            Deccan Chargers       0.38      0.50      0.43         6
             Delhi Capitals       0.45      0.33      0.38        15
              Gujarat Lions       0.75      1.00      0.86         3
       Kochi Tuskers Kerala       0.00      0.00      0.00         1
      Kolkata Knight Riders       0.62      0.53      0.57        19
             Mumbai Indians       0.58      0.68      0.62        22
              Pune Warriors       0.00      0.00      0.00         2
               Punjab Kings       0.57      0.75      0.65        16
           Rajasthan Royals       0.60      0.60      0.60        15
    Rising Pune Supergiants       0.60      1.00      0.75         3
Royal Challengers Bangalore       0.43      0.35      0.39        17
        Sunrisers Hyderabad       0.78      0.58      0.67        12

                   accuracy     

In [18]:
import os

# Create models folder if not present
os.makedirs("../models", exist_ok=True)


# Save Logistic Regression model
joblib.dump(
    lr_model,
    "../models/ipl_match_prediction_model.pkl"
)


# Save feature transformer
joblib.dump(
    encoder,
    "../models/encoder.pkl"
)


# Save winner label encoder
joblib.dump(
    winner_encoder,
    "../models/winner_encoder.pkl"
)


print("Model files saved successfully")

Model files saved successfully


In [19]:
# Example match

sample_match = pd.DataFrame({

    "team1": ["Mumbai Indians"],

    "team2": ["Chennai Super Kings"],

    "venue": ["Wankhede Stadium"],

    "toss_winner": ["Mumbai Indians"],

    "toss_decision": ["bat"]

})


sample_match

,team1,team2,venue,toss_winner,toss_decision
0,Mumbai Indians,Chennai Super Kings,Wankhede Stadium,Mumbai Indians,bat


In [20]:
sample_encoded = encoder.transform(
    sample_match
)

In [21]:
prediction = lr_model.predict(
    sample_encoded
)


winner = winner_encoder.inverse_transform(
    prediction
)


print("Predicted Winner:", winner[0])

Predicted Winner: Mumbai Indians
